# Spam Email Detection Template

This notebook builds several models (LogReg, LinearSVC, MultinomialNB) on two training sets combined,
evaluates them, then fits the best model on all training data and predicts labels for the test set.
Edit paths under **Config**. Outputs go to `results/`.


In [ ]:
# ==== Config (edit me) ====
LAST_NAME = "Arowolo"  # <-- replace with your last name
DATA_DIR = "../data" # folder containing TrainData*.csv / TrainLabel*.csv and TestData.csv

# Expected filenames (edit if your files differ)
TRAIN1_TEXT = "TrainData1.csv"     # must include a column named 'text' (or adjust below)
TRAIN1_LABEL = "TrainLabel1.csv"   # single column: 0=Ham, 1=Spam (or 'label' column)
TRAIN2_TEXT = "TrainData2.csv"
TRAIN2_LABEL = "TrainLabel2.csv"
TEST_TEXT = "TestData.csv"

TEXT_COLUMN = "text"     # change if your text column has a different name
LABEL_COLUMN = None      # if label CSV has a header name, set it; else leave None to use first column

RESULTS_DIR = "../results"
RANDOM_STATE = 42


In [ ]:
# ==== Imports ====
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.calibration import CalibratedClassifierCV


In [ ]:
# ==== Loaders ====
def load_text_csv(path: str, text_col: str):
    df = pd.read_csv(path)
    if text_col not in df.columns:
        # try without header
        df = pd.read_csv(path, header=None, names=[text_col])
    return df[text_col].astype(str)

def load_label_csv(path: str, label_col: str|None):
    df = pd.read_csv(path)
    if label_col is None:
        y = df.iloc[:,0]
    else:
        y = df[label_col]
    # normalize labels to {0,1} if they are strings like 'ham','spam'
    if y.dtype == object:
        y = y.str.lower().map({"ham":0, "spam":1}).fillna(y).astype(int)
    else:
        y = y.astype(int)
    return y


In [ ]:
# ==== Prepare combined training data ====
X1 = load_text_csv(os.path.join(DATA_DIR, TRAIN1_TEXT), TEXT_COLUMN)
y1 = load_label_csv(os.path.join(DATA_DIR, TRAIN1_LABEL), LABEL_COLUMN)

X2 = load_text_csv(os.path.join(DATA_DIR, TRAIN2_TEXT), TEXT_COLUMN)
y2 = load_label_csv(os.path.join(DATA_DIR, TRAIN2_LABEL), LABEL_COLUMN)

X_all = pd.concat([X1, X2], axis=0).reset_index(drop=True)
y_all = pd.concat([y1, y2], axis=0).reset_index(drop=True)

print(f"Combined training samples: {len(X_all)} (ham/spam counts: {(y_all==0).sum()}/{(y_all==1).sum()})")

# Load test
X_test = load_text_csv(os.path.join(DATA_DIR, TEST_TEXT), TEXT_COLUMN)
print(f"Test samples: {len(X_test)}")


In [ ]:
# ==== Build candidates ====
# We use TF-IDF with English stopwords; adjust max_features if needed.
tfidf = TfidfVectorizer(stop_words="english", max_features=30000, ngram_range=(1,2))

candidates = {
    "LogReg": Pipeline([("tfidf", tfidf), ("clf", LogisticRegression(max_iter=3000, n_jobs=None, random_state=RANDOM_STATE))]),
    "LinearSVC": Pipeline([("tfidf", tfidf), ("clf", LinearSVC(random_state=RANDOM_STATE))]),
    "MultinomialNB": Pipeline([("tfidf", tfidf), ("clf", MultinomialNB())]),
}


In [ ]:
# ==== Cross-validate and choose best ====
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = {}
for name, pipe in candidates.items():
    score = cross_val_score(pipe, X_all, y_all, cv=skf, scoring="f1", n_jobs=-1)  # F1 for spam detection
    cv_scores[name] = (score.mean(), score.std())
    print(f"{name}: F1 mean={score.mean():.4f} ± {score.std():.4f}")

best_name = max(cv_scores, key=lambda k: cv_scores[k][0])
best_pipeline = candidates[best_name]
print(f"Selected: {best_name}")


In [ ]:
# ==== Fit on all training data and evaluate on a hold-out split (sanity check) ====
X_tr, X_val, y_tr, y_val = train_test_split(X_all, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE)
best_pipeline.fit(X_tr, y_tr)
val_pred = best_pipeline.predict(X_val)

prec, rec, f1, _ = precision_recall_fscore_support(y_val, val_pred, average="binary", zero_division=0)
acc = accuracy_score(y_val, val_pred)
print(f"Hold-out Accuracy={acc:.4f}, Precision={prec:.4f}, Recall={rec:.4f}, F1={f1:.4f}")
print("\nClassification Report:\n", classification_report(y_val, val_pred, zero_division=0))


In [ ]:
# ==== Retrain on full data and predict test ====
best_pipeline.fit(X_all, y_all)
test_pred = best_pipeline.predict(X_test).astype(int)

# Save
os.makedirs(RESULTS_DIR, exist_ok=True)
out_path = os.path.join(RESULTS_DIR, f"{LAST_NAME}Spam.txt")
pd.Series(test_pred).to_csv(out_path, header=False, index=False)
print(f"Saved predictions to {out_path}")
